In [2]:
import pandas as pd
import numpy as np
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
from lightgbm import log_evaluation

In [10]:
df=pd.read_csv("../ecommerce_price_prediction-train.csv")

In [11]:
def preprocess(df):
    df = df.copy()
    
    # datetime
    df['capturedAt'] = pd.to_datetime(df['capturedAt'])
    df['day'] = df['capturedAt'].dt.day
    df['weekday'] = df['capturedAt'].dt.weekday
    df['is_weekend'] = df['weekday'].isin([5,6]).astype(int)
    
    # boolean mapping
    bool_cols = [
        'is_free_shipping', 'is_pre_order',
        'is_official_shop', 'is_verified',
        'is_preferred_plus_seller'
    ]
    for col in bool_cols:
        df[col] = df[col].map({'t': 1, 'f': 0})
    
    # brand
    df['brand'] = df['brand'].fillna('unknown')
    
    # stock sparse → indicator
    df['has_stock_info'] = df['normal_stock'].notnull().astype(int)
    
    # shop_response_rate
    df['shop_response_rate'] = df['shop_response_rate'].fillna(
        df['shop_response_rate'].median()
    )
    
    # avoid division by zero
    df['priceBeforeDiscount'] = df['priceBeforeDiscount'].replace(0, np.nan)
    
    # features
    df['discount_ratio'] = df['price'] / df['priceBeforeDiscount']
    df['discount_amount'] = df['priceBeforeDiscount'] - df['price']
    
    df['price_position'] = (
        (df['price'] - df['item_price_min']) /
        (df['item_price_max'] - df['item_price_min'] + 1)
    )
    
    df['log_rating_count'] = np.log1p(df['total_rating_count'])
    df['log_followers'] = np.log1p(df['shop_follower_count'])
    
    df['shop_score'] = df['shop_rating'] * df['shop_response_rate']
    
    return df

Train / Validation Split

In [12]:
df = preprocess(df)

df = df.sort_values('capturedAt')

# last day as validation
split_date = df['capturedAt'].quantile(0.9)

train_df = df[df['capturedAt'] < split_date]
val_df = df[df['capturedAt'] >= split_date]

Define Features

In [13]:
target = 'price'

features = [
    'shopId', 'itemId', 'modelId', 'cat_id',
    'priceBeforeDiscount', 'promotionId',
    'raw_discount', 'show_discount',
    'item_price_min', 'item_price_max',
    'review_rating', 'total_rating_count',
    'cmt_count', 'shop_rating',
    'shop_response_rate', 'shop_follower_count',
    'is_free_shipping', 'is_pre_order',
    'is_official_shop', 'is_verified',
    'is_preferred_plus_seller',
    
    # engineered
    'discount_ratio', 'discount_amount',
    'price_position', 'log_rating_count',
    'log_followers', 'shop_score',
    'weekday', 'is_weekend', 'has_stock_info'
]

In [14]:
model = LGBMRegressor(
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=-1,
    num_leaves=64,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)



model.fit(
    train_df[features],
    np.log1p(train_df[target]),
    eval_set=[(val_df[features], np.log1p(val_df[target]))],
    eval_metric='rmse',
    callbacks=[log_evaluation(100)]  # ✅ correct replacement
)

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.037016 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 4740
[LightGBM] [Info] Number of data points in the train set: 275603, number of used features: 29
[LightGBM] [Info] Start training from score 17.004101
[100]	valid_0's rmse: 0.0410505	valid_0's l2: 0.00168514
[200]	valid_0's rmse: 0.0278477	valid_0's l2: 0.000775493
[300]	valid_0's rmse: 0.0246475	valid_0's l2: 0.000607501
[400]	valid_0's rmse: 0.0230258	valid_0's l2: 0.00053019
[500]	valid_0's rmse: 0.0221838	valid_0's l2: 0.00049212
[600]	valid_0's rmse: 0.0216156	valid_0's l2: 0.000467234
[700]	valid_0's rmse: 0.0212564	valid_0's l2: 0.000451836
[800]	valid_0's rmse: 0.0209676	valid_0's l2: 0.000439642
[900]	valid_0's rmse: 0.0207646	valid_0's l2: 0.000431168
[1000]	valid_0's rmse: 0.0206295	valid_0's l2: 0.000425577


LGBMRegressor(colsample_bytree=0.8, learning_rate=0.05, n_estimators=1000,
              num_leaves=64, random_state=42, subsample=0.8)

In [15]:
val_pred_log = model.predict(val_df[features])
val_pred = np.expm1(val_pred_log)

mae = mean_absolute_error(val_df[target], val_pred)
rmse = np.sqrt(mean_squared_error(val_df[target], val_pred))
mape = np.mean(np.abs((val_df[target] - val_pred) / val_df[target])) * 100

print(f"MAE: {mae:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"MAPE: {mape:.2f}%")

MAE: 167031.35
RMSE: 1315456.89
MAPE: 0.37%


In [13]:
import joblib

joblib.dump(model, "models/global_model.pkl")

['models/global_model.pkl']

Callibration Framework

In [18]:
test_df=pd.read_csv("../ecommerce_price_prediction-test-3-days.csv")
test_df = preprocess(test_df)

In [19]:
test_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25900 entries, 0 to 25899
Data columns (total 36 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   capturedAt                25900 non-null  datetime64[ns]
 1   shopId                    25900 non-null  int64         
 2   itemId                    25900 non-null  int64         
 3   modelId                   25900 non-null  int64         
 4   price                     300 non-null    float64       
 5   priceBeforeDiscount       119 non-null    float64       
 6   promotionId               300 non-null    float64       
 7   cat_id                    300 non-null    float64       
 8   stock                     7 non-null      float64       
 9   normal_stock              7 non-null      float64       
 10  raw_discount              300 non-null    float64       
 11  show_discount             300 non-null    float64       
 12  brand             

In [20]:
test_df['pred_base_log'] = model.predict(test_df[features])
test_df['pred_base'] = np.expm1(test_df['pred_base_log'])

In [21]:
test_df.head()

,capturedAt,shopId,itemId,modelId,price,priceBeforeDiscount,promotionId,cat_id,stock,normal_stock,...,is_weekend,has_stock_info,discount_ratio,discount_amount,price_position,log_rating_count,log_followers,shop_score,pred_base_log,pred_base
0,2025-03-22 04:27:05.325,1009757562,27904817781,139002028392,NaN,NaN,NaN,NaN,NaN,NaN,...,1,0,NaN,NaN,NaN,NaN,NaN,NaN,12.158528,190712.586465
1,2025-03-22 04:27:05.325,1009757562,27904817781,139002028392,NaN,NaN,NaN,NaN,NaN,NaN,...,1,0,NaN,NaN,NaN,NaN,NaN,NaN,12.158528,190712.586465
2,2025-03-22 04:27:05.325,1009757562,27904817781,241162534603,NaN,NaN,NaN,NaN,NaN,NaN,...,1,0,NaN,NaN,NaN,NaN,NaN,NaN,12.174135,193712.326381
3,2025-03-22 04:27:05.325,1009757562,27904817781,241162534603,NaN,NaN,NaN,NaN,NaN,NaN,...,1,0,NaN,NaN,NaN,NaN,NaN,NaN,12.174135,193712.326381
4,2025-03-22 04:27:05.325,1009757562,27904817781,241162534604,NaN,NaN,NaN,NaN,NaN,NaN,...,1,0,NaN,NaN,NaN,NaN,NaN,NaN,12.174135,193712.326381


In [23]:
#Predict base for anchor rows
anchor_df = test_df[test_df['price'].notnull()].copy()

In [24]:
#compute ratio
anchor_df['ratio'] = anchor_df['price'] / anchor_df['pred_base']

In [26]:
#Build Callibration
#Global Shift
global_shift = anchor_df['ratio'].median()
#Category Shift
cat_shift = (
    anchor_df.groupby('cat_id')['ratio']
    .median()
    .to_dict()
)

In [27]:
#Only trust shop if enough anchors.
shop_stats = anchor_df.groupby('shopId')['ratio'].agg(['median', 'count']).reset_index()

shop_shift = {
    row['shopId']: row['median']
    for _, row in shop_stats.iterrows()
    if row['count'] >= 3
}

In [28]:
#Apply Smart Calliration
def calibrate_row(row):
    if row['shopId'] in shop_shift:
        shift = shop_shift[row['shopId']]
    elif row['cat_id'] in cat_shift:
        shift = cat_shift[row['cat_id']]
    else:
        shift = global_shift
        
    return row['pred_base'] * shift


In [29]:
test_df['pred_final'] = test_df.apply(calibrate_row, axis=1)

In [30]:
test_df.loc[test_df['price'].notnull(), 'pred_final'] = test_df['price']

In [31]:
submission = test_df.copy()
submission['price'] = submission['pred_final'].round().astype(int)

In [36]:
def blended_calibrate(row):
    
    g = global_shift
    
    c = cat_shift.get(row['cat_id'], g)
    
    s = shop_shift.get(row['shopId'], c)
    
    final_shift = (
        0.5 * g +
        0.3 * c +
        0.2 * s
    )
    
    return row['pred_base'] * final_shift